# Data Pre-processing Notebook
This notebook carries out the necessary pre-processing of the datasets to prepare it for the evaluation of the BLIP and BERT models. It additionally downlaods any images required for the BLIP models as well.

In [1]:
### DEPENDENCIES ###
import pandas as pd
import numpy as np
import re
import requests
import os
import time
from tqdm import tqdm
from PIL import Image
import io

# Loading and Cleaning the Fakeddit Dataset 

The following process cleans up the Fakeddit dataset, keeping only the necessary columns for the next stage and ensures there are no misleading 

In [2]:
# Loading the dataset tsv file
input_tsv_path = 'multimodal_test_public.tsv'
print("Loading Fakeddit TSV Dataset")
df_full = pd.read_csv(input_tsv_path, sep = '\t')
print(f"Original Rows: {len(df_full)}")

# Keeping only the relevant columns
df_full = df_full[['id', 'clean_title', 'image_url', '6_way_label']]

# Droping any missing data
df_full = df_full.dropna(subset = ['clean_title', '6_way_label', 'image_url'])

# Converting spaces into NaN and dropping them
df_full['clean_title'] = df_full['clean_title'].replace(r'^\s*$', np.nan, regex = True)
df_full = df_full.dropna(subset = ['clean_title'])

"""
text_cleaning Function
Removes any URLs and any leading/trailing whitespaces
"""
def text_cleaning(text):

    if not isinstance(text, str):
        return ""

    # URL Removal
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags = re.MULTILINE)

    # Strip leading/trailing whitespace
    return text.strip()

df_full['clean_title'] = df_full['clean_title'].apply(text_cleaning)

# Validating that there is no empty strings again as precaution
df_full['clean_title'] = df_full['clean_title'].replace(r'^\s*$', np.nan, regex = True)
df_full = df_full.dropna(subset = ['clean_title'])

print(f"\nCleaning Done! \nFinal Row Count: {len(df_full)}")

Loading Fakeddit TSV Dataset
Original Rows: 59319

Cleaning Done! 
Final Row Count: 58983


# Fakeddit Pre-processing for BLIP

In this process, I will read in the fakeddit dataset with the aim of extracting a 2000 row 50/50, 

In [28]:
# Defining Variable Configurations
image_folder = 'fakeddit_images'
blip_output_csv = 'fakeddit_blip_50_50_test.csv'

# Setting Target Size per Class
target_per_class= 1000

# Disgusing Script as Browser Attempts to Avoid Error 429
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Creating the image folder if it does not exist
os.makedirs(image_folder, exist_ok = True)

# Filtering the dataset to only include True and False Connection
df_blip_candidates = df_full[df_full['6_way_label'].isin([0, 2])].copy()
df_blip_candidates = df_blip_candidates.sample(frac = 1, random_state = 42).reset_index(drop = True)

print(f"Total True/False Connection candidates available in the raw dataset: {len(df_blip_candidates)}")

Total True/False Connection candidates available in the raw dataset: 34662


In [25]:
valid_true_rows = []
valid_false_conn_rows = []

print(f"\nDownloading VALID images until there are 1000 images per class")

for index, row in tqdm(df_blip_candidates.iterrows(), total = len(df_blip_candidates)):
    
    # Check if target has been hit for both class, if it has been hit exit the loop
    if len(valid_true_rows) >= target_per_class and len(valid_false_conn_rows) >= target_per_class:
        print("\nTarget reached for both classes! Stopping downloads.")
        break

    label = row['6_way_label']
    
    # Check if there is sufficient data per classs
    if label == 0 and len(valid_true_rows) >= target_per_class:
        continue
    if label == 2 and len(valid_false_conn_rows) >= target_per_class:
        continue

    img_url = row.get('image_url')
    img_id = str(row.get('id'))
    
    # Checking for invalid rows, skip if invalid
    if pd.isna(img_url) or not str(img_url).startswith('http'):
        continue

    # Variables for retrying failed downloads
    max_retries = 3
    retry_delay = 2 

    # For Loop to Attempt Download
    for attempt in range(max_retries):
        try:
            # Request the image
            response = requests.get(img_url, headers=HEADERS, timeout=5)

            # If response code 200 (OK)
            if response.status_code == 200:
                # Verify it is a readable image file
                image_bytes = io.BytesIO(response.content)
                img = Image.open(image_bytes)
                img.verify() 
                
                # Save the image to the folder
                file_path = os.path.join(image_folder, f"{img_id}.jpg")
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                
                # Record the local path to the file and add it to the data
                row['local_image_path'] = file_path

                # Append the adjusted row to their respective valid label list
                if label == 0:
                    valid_true_rows.append(row)
                elif label == 2:
                    valid_false_conn_rows.append(row)
                
                # Pause to avoid rate limiting
                time.sleep(0.1) 
                break 

            # If response code 429 (Too many requests)
            elif response.status_code == 429:
                # Wait and try again due to rate limitation
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2 
                else:
                    pass
            else:
                # For if link or image not found
                break 

        except Exception as e:
            break

# Finalize the Dataset
print("\nDownlaod Completed for BLIP Dataset")
print(f"Successfully downloaded True images: {len(valid_true_rows)}")
print(f"Successfully downloaded False Connection images: {len(valid_false_conn_rows)}")

 10%|███████▎                                                                   | 3364/34662 [22:38<3:30:36,  2.48it/s]


Target reached for both classes! Stopping downloads.

Downlaod Completed for BLIP Dataset
Successfully downloaded True images: 1000
Successfully downloaded False Connection images: 1000


In [29]:
# Combine the lists into a DataFrame
df_final_blip = pd.DataFrame(valid_true_rows + valid_false_conn_rows)

# Shuffle to ensure data is not stuck together
df_final_blip = df_final_blip.sample(frac = 1, random_state = 42).reset_index(drop = True)

# Save the final CSV
df_final_blip.to_csv(blip_output_csv, index = False)
print(f"\nDataset reduced to 50/50 True/False Connection and saved to '{blip_output_csv}'")


Dataset reduced to 50/50 True/False Connection and saved to 'fakeddit_blip_50_50_test.csv'


# ISOT Dataset Read and Clean



In [13]:
# Reading in the welfake dataset
df_fake = pd.read_csv('Fake.csv')
df_real = pd.read_csv('True.csv')

# Labelling the Datasets
df_fake['label'] = 0 
df_real['label'] = 1 

# Combining the datasets
df_isot = pd.concat([df_fake, df_real])[['title', 'text', 'label']].copy()
print(f"Original Rows: {len(df_isot)}")

# Drop rows with no title or text
df_isot = df_isot.dropna(subset=['title', 'text'])

# Converting whitespaces to NaN and dropping the rows
df_isot['title'] = df_isot['title'].replace(r'^\s*$', np.nan, regex = True)
df_isot['text']  = df_isot['text'].replace(r'^\s*$', np.nan, regex = True)
df_isot = df_isot.dropna(subset=['title', 'text'])

# Apply text cleaning function to both columns
df_isot['title'] = df_isot['title'].apply(text_cleaning)
df_isot['text']  = df_isot['text'].apply(text_cleaning)

# Validate no empty strings again as precaution
df_isot['title'] = df_isot['title'].replace(r'^\s*$', np.nan, regex = True)
df_isot['text']  = df_isot['text'].replace(r'^\s*$', np.nan, regex = True)
df_isot = df_isot.dropna(subset=['title', 'text'])

# Combining title and text into a single column
df_isot['combined'] = df_isot['title'] + ' ' + df_isot['text']
print(f"\nCleaning Done! \nFinal Row Count: {len(df_isot)}")

Original Rows: 44898

Cleaning Done! 
Final Row Count: 44183


In [15]:
# Retrieving label distribution
print(df_isot['label'].value_counts(normalize = True) * 100)

# Defining the ratio as variable
bert_ratios = df_isot['label'].value_counts(normalize = True)
df_bert_final = pd.DataFrame()

# Stratified sample to 2000 rows preserve original ratio
for label, ratio in bert_ratios.items():
    n_needed = int(round(2000 * ratio))
    sampled = df_isot[df_isot['label'] == label].sample(n = n_needed, random_state = 42)
    df_bert_final = pd.concat([df_bert_final, sampled])

# Shuffling the data
df_bert_final = df_bert_final.sample(frac = 1, random_state = 42).reset_index(drop = True)

# Finalizing the dataset
df_bert_final.to_csv('isot_bert_test.csv', index = False)
print(f"Saved {len(df_bert_final)} rows to welfake_bert_test.csv")
print(df_bert_final['label'].value_counts())

label
0    51.528869
1    48.471131
Name: proportion, dtype: float64
Saved 2000 rows to welfake_bert_test.csv
label
0    1031
1     969
Name: count, dtype: int64
